In [ ]:
import jax
import jax.numpy as jnp
import functions
import pybamm
import numpy as np

In [ ]:
# Set random keys
main_key = jax.random.PRNGKey(0)
key_train, key_test = jax.random.split(main_key)

params_bat = pybamm.ParameterValues("Chen2020")

# Hyperparameters
num_train = 2200
num_test = 220

C = params_bat["Nominal cell capacity [A.h]"]
t_max = 1800
num_samples_I = 75
num_samples_c0 = 20

t = np.linspace(0, t_max, num_samples_I)
r = np.linspace(0, 1, num_samples_c0)

In [ ]:
spm = pybamm.lithium_ion.SPM()
Ran = params_bat["Negative particle radius [m]"]
Rca = params_bat["Positive particle radius [m]"]

def get_targets(I_samples, soc=0.5):
    set_targets = []
    set_c0 = []

    I_pruned = []
    
    for id, I_func in enumerate(I_samples):
        # Update parameter
        params_bat["Current function [A]"] = pybamm.Interpolant(t, -1.* I_func, pybamm.t)
        
        # Re-create the simulation with updated parameters
        sim = pybamm.Simulation(spm, parameter_values=params_bat)
        
        sol = sim.solve(initial_soc=soc, t_eval=t)
        c0 = sol["Negative particle concentration"].entries[:,0,0]
        cn_target = sol["Negative particle concentration"].entries[:,0,:]

        if cn_target.shape[-1] != num_samples_I:
            continue
        else:
            I_pruned.append(I_func)
        
        set_targets.append(cn_target)
        set_c0.append(c0)

    
    return set_targets, set_c0, I_pruned


# def generate_data(key, num):
#     keys = jax.random.split(key, num)
#     func_I = []
#     for k in keys:
#         # Generate a random current function
#         I_func = functions.GaussianRFCurrent(k, C, t_max)
#         func_I.append(I_func(t))
#     return func_I

def generate_data(key, num):
    keys = jax.random.split(key, num)
    train_I = []
    for k in keys:
        # Sample a random amplitude value for the triangle current
        value = jax.random.uniform(k, shape=(), minval=-1, maxval=1)
        I_func = functions.TriangleCurrent(value * C)
        train_I.append(I_func(t))
    return train_I

# def generate_data(key, num_samples):
#     keys = jax.random.split(key, num_samples)
#     set_I = []
#     for k in keys:
#         # Generate a random constant current value between -1C and 1C
#         rand_value = jax.random.uniform(k, shape=(), minval=-1, maxval=1)
        
#         # Create a constant current function with that value
#         I_func = functions.ConstantCurrent(rand_value*C)
#         # Evaluate at times t and append
#         set_I.append(I_func(t))
#     return set_I

In [ ]:
train_I = generate_data(key_train, num_train)
test_I = generate_data(key_test, num_test)
train_cn, train_c0, train_I = get_targets(train_I)
test_cn, test_c0, test_I = get_targets(test_I)

train_I = jnp.array(train_I)
test_I = jnp.array(test_I)
train_cn = jnp.array(train_cn)
test_cn = jnp.array(test_cn)
train_c0 = jnp.array(train_c0)
test_c0 = jnp.array(test_c0)